# Piecewise McCormick Gap Reduction (Table 4)

This notebook reproduces the piecewise McCormick gap reduction numbers for Table 4 of the manuscript.
For each function, we compare the standard McCormick relaxation gap (cc - cv) with the
piecewise McCormick gap using k=8 uniform partitions, and report the percentage gap reduction.

In [1]:
import os

os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["JAX_ENABLE_X64"] = "1"

import jax
import jax.numpy as jnp
from discopt._jax.mccormick import relax_exp, relax_log, relax_square
from discopt._jax.piecewise_mccormick import (
    piecewise_relax_exp,
    piecewise_relax_log,
    piecewise_relax_square,
)

In [2]:
N_POINTS = 10_000
K = 8  # number of piecewise partitions


def random_points(key, lb, ub, n=N_POINTS):
    """Generate n random points uniformly in [lb, ub]."""
    return lb + (ub - lb) * jax.random.uniform(key, shape=(n,), dtype=jnp.float64)


def compute_gap_reduction(std_relax_fn, pw_relax_fn, true_fn, lb, ub, key):
    """Compute standard gap, piecewise gap, and gap reduction percentage.

    Args:
        std_relax_fn: function(x) -> (cv, cc) for standard McCormick
        pw_relax_fn: function(x) -> (cv, cc) for piecewise McCormick
        true_fn: the true function f(x)
        lb, ub: interval bounds
        key: JAX PRNG key

    Returns:
        (std_gap, pw_gap, reduction_pct)
    """
    x = random_points(key, lb, ub)

    # Standard McCormick gap
    std_cv, std_cc = jax.vmap(std_relax_fn)(x)
    std_gap = float(jnp.mean(std_cc - std_cv))

    # Piecewise McCormick gap
    pw_cv, pw_cc = jax.vmap(pw_relax_fn)(x)
    pw_gap = float(jnp.mean(pw_cc - pw_cv))

    # Gap reduction
    reduction = 1.0 - pw_gap / max(std_gap, 1e-15)

    return std_gap, pw_gap, reduction * 100.0

In [3]:
# Define test cases: (name, true_fn, std_relax, pw_relax, lb, ub)
test_cases = [
    {
        "name": "exp(x)",
        "interval": "[-2, 3]",
        "lb": -2.0,
        "ub": 3.0,
        "std_fn": lambda xi: relax_exp(xi, -2.0, 3.0),
        "pw_fn": lambda xi: piecewise_relax_exp(xi, -2.0, 3.0, k=K),
    },
    {
        "name": "log(x)",
        "interval": "[0.5, 8]",
        "lb": 0.5,
        "ub": 8.0,
        "std_fn": lambda xi: relax_log(xi, 0.5, 8.0),
        "pw_fn": lambda xi: piecewise_relax_log(xi, 0.5, 8.0, k=K),
    },
    {
        "name": "x^2",
        "interval": "[-3, 4]",
        "lb": -3.0,
        "ub": 4.0,
        "std_fn": lambda xi: relax_square(xi, -3.0, 4.0),
        "pw_fn": lambda xi: piecewise_relax_square(xi, -3.0, 4.0, k=K),
    },
]

# Compute results
results = []
for i, tc in enumerate(test_cases):
    key = jax.random.PRNGKey(42 + i)
    std_gap, pw_gap, reduction = compute_gap_reduction(
        tc["std_fn"], tc["pw_fn"], None, tc["lb"], tc["ub"], key
    )
    results.append(
        {
            "name": tc["name"],
            "interval": tc["interval"],
            "std_gap": std_gap,
            "pw_gap": pw_gap,
            "reduction": reduction,
        }
    )

In [4]:
# Print Table 4
print("Table 4: Piecewise McCormick Gap Reduction (k=8 uniform partitions)")
print("=" * 75)
print(f"{'Function':<12} {'Interval':<12} {'Std Gap':>12} {'PW Gap (k=8)':>14} {'Reduction':>12}")
print("-" * 75)
for r in results:
    print(
        f"{r['name']:<12} {r['interval']:<12}"
        f" {r['std_gap']:>12.6f} {r['pw_gap']:>14.6f} {r['reduction']:>11.1f}%"
    )
print("=" * 75)
print(f"\nN = {N_POINTS} random sample points per function.")
print("Gap = mean(cc - cv) over sample points.")
print("Reduction = (1 - pw_gap / std_gap) * 100%.")

Table 4: Piecewise McCormick Gap Reduction (k=8 uniform partitions)
Function     Interval          Std Gap   PW Gap (k=8)    Reduction
---------------------------------------------------------------------------
exp(x)       [-2, 3]          6.180083       0.128979        97.9%
log(x)       [0.5, 8]         0.565990       0.017134        97.0%
x^2          [-3, 4]          8.098263       0.127233        98.4%

N = 10000 random sample points per function.
Gap = mean(cc - cv) over sample points.
Reduction = (1 - pw_gap / std_gap) * 100%.
